# Fase 4 — Execution Engine e Portafoglio
## Progetto AI_microtradingAUX

Questo notebook documenta in dettaglio l'implementazione della Fase 4, dedicata alla simulazione realistica dell'esecuzione degli ordini e alla gestione del portafoglio per l'agente di trading su XAU/USD M1.

**Durata indicativa:** 3-4 giorni

**Obiettivo:** trasformare una decisione di posizione (long/flat/short) in un trade eseguibile con costi realistici (spread, commissioni, slippage), e tracciare correttamente capitale, equity e P&L realizzato/non realizzato barra per barra.

### Vincolo di partenza

Il dataset disponibile ha un solo prezzo OHLC per barra, senza Bid/Ask separati. Lo spread di mercato è stato quindi **simulato** attorno al prezzo mid, invece di essere letto direttamente da colonne Bid/Ask assenti nel dataset.

### File prodotti

| File | Ruolo |
|---|---|
| `src/gold_rl/execution.py` | Prezzo di esecuzione, spread, slippage, commissioni, turnover |
| `src/gold_rl/portfolio.py` | Stato del portafoglio: cash, equity, posizione, P&L realizzato/non realizzato |
| `src/gold_rl/simulation.py` | Collegamento con la Fase 3: produce `current_position` e `unrealized_return` |
| `tests/test_execution.py` | Test unitari sull'Execution Engine |
| `tests/test_portfolio.py` | Test sul portafoglio, incluso il criterio di completamento |
| `tests/test_state_frame_integration.py` | Test di integrazione Fase 3 ↔ Fase 4 |

### Criterio di completamento

> Una sequenza di azioni costruita manualmente deve produrre esattamente l'equity attesa.

Verificato con **18/18 test passati** nell'intero progetto, inclusi 3 test dedicati esplicitamente al calcolo manuale dell'equity attesa con tolleranza `rel_tol=1e-12` (precisione floating-point).


## 1. Scelte progettuali preliminari

Prima di scrivere il codice sono state fissate alcune convenzioni, necessarie per rendere il sistema deterministico e testabile manualmente.

### Spazio delle posizioni

Coerentemente con `config.yaml` (`positions: [-1, 0, 1]`), la posizione netta ammessa è discreta:
- `-1` = short 1 unità nozionale
- `0` = flat
- `1` = long 1 unità nozionale

### Turnover

Misura quanta esposizione viene cambiata in una transizione:

\\[ \\text{turnover} = |\\text{target\\_position} - \\text{previous\\_position}| \\]

Esempi: `0→1` turnover 1; `1→-1` turnover 2 (chiusura long + apertura short in un solo movimento).

### Simulazione dello spread senza Bid/Ask

Poiché il dataset fornisce solo un prezzo mid, lo spread è stato simulato applicando una penalità simmetrica al mid price in funzione del lato dell'operazione:
- acquisto (`trade_size > 0`): si paga sopra il mid
- vendita (`trade_size < 0`): si riceve sotto il mid

Questo replica l'effetto economico di un vero spread Bid/Ask, usando solo `spread_rate` da `config.yaml`.

### Separazione realized / unrealized P&L

- **Realized P&L**: aggiornato solo quando una posizione viene chiusa o invertita.
- **Unrealized P&L**: ricalcolato ogni barra sulla posizione aperta, confrontando `entry_price` con il mid corrente.
- **Equity**: definita come `cash + position * mid_price`, la formulazione più robusta da verificare manualmente.


## 2. Modulo `execution.py`

### Configurazione dei costi

`ExecutionCostsConfig` raccoglie i quattro parametri economici già presenti in `config.yaml`: `spread_rate`, `commission_rate`, `slippage_rate`, `turnover_penalty`.


In [1]:

from __future__ import annotations
from dataclasses import dataclass


@dataclass(frozen=True)
class ExecutionCostsConfig:
    spread_rate: float
    commission_rate: float
    slippage_rate: float
    turnover_penalty: float


@dataclass(frozen=True)
class ExecutionResult:
    timestamp: object
    mid_price: float
    execution_price: float
    previous_position: int
    target_position: int
    trade_size: int
    turnover: float
    spread_cost: float
    slippage_cost: float
    commission_cost: float
    turnover_cost: float
    total_cost: float


### Validazione e calcolo di trade size e turnover

`validate_position_transition` blocca posizioni fuori dal dominio consentito prima di qualunque calcolo economico.


In [2]:

def validate_position_transition(
    previous_position: int,
    target_position: int,
    allowed_positions: tuple[int, ...] = (-1, 0, 1),
) -> None:
    if previous_position not in allowed_positions:
        raise ValueError(f"Posizione precedente non valida: {previous_position}")
    if target_position not in allowed_positions:
        raise ValueError(f"Posizione target non valida: {target_position}")


def get_trade_size(previous_position: int, target_position: int) -> int:
    return target_position - previous_position


def get_turnover(previous_position: int, target_position: int) -> float:
    return float(abs(target_position - previous_position))


### Prezzo di esecuzione

Il cuore della simulazione dello spread: il prezzo effettivo è sempre peggiore del mid, nella direzione sfavorevole al trader.


In [3]:

def get_execution_price(
    mid_price: float,
    trade_size: int,
    spread_rate: float,
    slippage_rate: float,
) -> float:
    if mid_price <= 0:
        raise ValueError("mid_price deve essere positivo.")

    if trade_size == 0:
        return float(mid_price)

    half_spread = spread_rate / 2.0

    if trade_size > 0:
        # acquisto: si paga sopra il mid (spread + slippage avverso)
        return float(mid_price * (1.0 + half_spread + slippage_rate))

    # vendita: si riceve sotto il mid
    return float(mid_price * (1.0 - half_spread - slippage_rate))


In [4]:

# Verifica pratica: un acquisto a mid=100 con spread 0.02% e slippage 0.005%
# deve costare 100 * (1 + 0.0001 + 0.00005) = 100.015
prezzo_buy = get_execution_price(mid_price=100.0, trade_size=1, spread_rate=0.0002, slippage_rate=0.00005)
prezzo_sell = get_execution_price(mid_price=100.0, trade_size=-1, spread_rate=0.0002, slippage_rate=0.00005)
print(f"Prezzo di acquisto: {prezzo_buy}  (mid=100, penalizzato verso l'alto)")
print(f"Prezzo di vendita:  {prezzo_sell}  (mid=100, penalizzato verso il basso)")


Prezzo di acquisto: 100.01500000000001  (mid=100, penalizzato verso l'alto)
Prezzo di vendita:  99.985  (mid=100, penalizzato verso il basso)


### Calcolo dei costi di transazione

I quattro costi sono tenuti separati per trasparenza e debug, tutti proporzionali al nozionale scambiato (`mid_price * turnover`).


In [5]:

def calculate_transaction_costs(
    mid_price: float,
    turnover: float,
    config: ExecutionCostsConfig,
) -> tuple[float, float, float, float, float]:
    notional = mid_price * turnover

    spread_cost = notional * (config.spread_rate / 2.0)
    slippage_cost = notional * config.slippage_rate
    commission_cost = notional * config.commission_rate
    turnover_cost = notional * config.turnover_penalty

    total_cost = spread_cost + slippage_cost + commission_cost + turnover_cost

    return (
        float(spread_cost), float(slippage_cost),
        float(commission_cost), float(turnover_cost), float(total_cost),
    )


def execute_order(
    timestamp: object,
    mid_price: float,
    previous_position: int,
    target_position: int,
    config: ExecutionCostsConfig,
) -> ExecutionResult:
    """Orchestratore: valida, calcola trade size/turnover, prezzo e costi."""
    validate_position_transition(previous_position=previous_position, target_position=target_position)

    trade_size = get_trade_size(previous_position=previous_position, target_position=target_position)
    turnover = get_turnover(previous_position=previous_position, target_position=target_position)

    execution_price = get_execution_price(
        mid_price=mid_price, trade_size=trade_size,
        spread_rate=config.spread_rate, slippage_rate=config.slippage_rate,
    )

    spread_cost, slippage_cost, commission_cost, turnover_cost, total_cost = calculate_transaction_costs(
        mid_price=mid_price, turnover=turnover, config=config,
    )

    return ExecutionResult(
        timestamp=timestamp, mid_price=float(mid_price), execution_price=execution_price,
        previous_position=previous_position, target_position=target_position, trade_size=trade_size,
        turnover=turnover, spread_cost=spread_cost, slippage_cost=slippage_cost,
        commission_cost=commission_cost, turnover_cost=turnover_cost, total_cost=total_cost,
    )


In [6]:

# Dimostrazione: apertura di un long da flat, con costi scomposti
config_demo = ExecutionCostsConfig(spread_rate=0.0002, commission_rate=0.0001, slippage_rate=0.00005, turnover_penalty=0.0001)

risultato = execute_order(timestamp="2024-01-01 10:00", mid_price=100.0, previous_position=0, target_position=1, config=config_demo)

print(f"trade_size={risultato.trade_size}  turnover={risultato.turnover}")
print(f"execution_price={risultato.execution_price}")
print(f"spread_cost={risultato.spread_cost:.6f}  slippage_cost={risultato.slippage_cost:.6f}")
print(f"commission_cost={risultato.commission_cost:.6f}  turnover_cost={risultato.turnover_cost:.6f}")
print(f"total_cost={risultato.total_cost:.6f}")


trade_size=1  turnover=1.0
execution_price=100.01500000000001
spread_cost=0.010000  slippage_cost=0.005000
commission_cost=0.010000  turnover_cost=0.010000
total_cost=0.035000


## 3. Modulo `portfolio.py`

### Stato del portafoglio

`PortfolioState` è un oggetto immutabile (`frozen=True`) che rappresenta lo stato completo del conto in un dato istante: cassa, equity, posizione, prezzo medio di ingresso, P&L realizzato/non realizzato, costi e turnover cumulati.


In [7]:

from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioState:
    cash: float
    equity: float
    position: int
    entry_price: float | None
    realized_pnl: float
    unrealized_pnl: float
    cumulative_costs: float
    cumulative_turnover: float


def initialize_portfolio(initial_balance: float) -> PortfolioState:
    return PortfolioState(
        cash=float(initial_balance), equity=float(initial_balance),
        position=0, entry_price=None, realized_pnl=0.0, unrealized_pnl=0.0,
        cumulative_costs=0.0, cumulative_turnover=0.0,
    )


### Calcolo del P&L non realizzato e mark-to-market

`unrealized_pnl` confronta sempre l'entry price con il mid price corrente, con segno opposto per long e short.


In [8]:

def calculate_unrealized_pnl(position: int, entry_price, mid_price: float) -> float:
    if position == 0 or entry_price is None:
        return 0.0
    if position > 0:
        return float((mid_price - entry_price) * position)
    return float((entry_price - mid_price) * abs(position))


def mark_to_market_value(position: int, mid_price: float) -> float:
    return float(position * mid_price)


### Aggiornamento del portafoglio

Questa è la funzione centrale della Fase 4. Gestisce quattro casi distinti in base alla transizione di posizione:

1. **Da flat ad una posizione**: si apre una nuova posizione, `entry_price` viene fissato al prezzo di esecuzione.
2. **Chiusura verso flat**: si realizza il P&L della posizione chiusa, `entry_price` torna a `None`.
3. **Mantenimento della stessa direzione**: nessun realized P&L, solo aggiornamento dell'unrealized.
4. **Flip diretto (long→short o short→long)**: si realizza il P&L della posizione precedente e si apre immediatamente la nuova, con nuovo `entry_price`.

La cassa si aggiorna sempre con la stessa formula elegante: `cash -= execution_price * trade_size` seguita da `cash -= total_cost`. Se `trade_size > 0` (acquisto) la cassa scende; se `trade_size < 0` (vendita) sottraendo un numero negativo la cassa sale.


In [9]:

def update_portfolio(previous_state: PortfolioState, execution, mid_price: float) -> PortfolioState:
    previous_position = previous_state.position
    target_position = execution.target_position
    trade_size = execution.trade_size

    cash = previous_state.cash
    realized_pnl = previous_state.realized_pnl
    cumulative_costs = previous_state.cumulative_costs + execution.total_cost
    cumulative_turnover = previous_state.cumulative_turnover + execution.turnover

    previous_entry_price = previous_state.entry_price
    new_entry_price = previous_entry_price

    if trade_size != 0:
        cash -= execution.execution_price * trade_size
        cash -= execution.total_cost

    if previous_position == 0:
        if target_position != 0:
            new_entry_price = execution.execution_price

    elif previous_position > 0:
        if target_position == 0:
            realized_pnl += (execution.execution_price - previous_entry_price) * previous_position
            new_entry_price = None
        elif target_position > 0:
            if target_position != previous_position:
                new_entry_price = execution.execution_price
        else:
            realized_pnl += (execution.execution_price - previous_entry_price) * previous_position
            new_entry_price = execution.execution_price

    elif previous_position < 0:
        if target_position == 0:
            realized_pnl += (previous_entry_price - execution.execution_price) * abs(previous_position)
            new_entry_price = None
        elif target_position < 0:
            if target_position != previous_position:
                new_entry_price = execution.execution_price
        else:
            realized_pnl += (previous_entry_price - execution.execution_price) * abs(previous_position)
            new_entry_price = execution.execution_price

    unrealized_pnl = calculate_unrealized_pnl(position=target_position, entry_price=new_entry_price, mid_price=mid_price)
    equity = cash + mark_to_market_value(position=target_position, mid_price=mid_price)

    return PortfolioState(
        cash=float(cash), equity=float(equity), position=int(target_position),
        entry_price=new_entry_price, realized_pnl=float(realized_pnl), unrealized_pnl=float(unrealized_pnl),
        cumulative_costs=float(cumulative_costs), cumulative_turnover=float(cumulative_turnover),
    )


## 4. Verifica manuale del criterio di completamento

### Round-trip long: `0 → 1 → 0`

Si simula un acquisto a mid=100 seguito da una vendita a mid=101, e si confronta l'equity finale con un calcolo manuale indipendente della stessa sequenza.


In [10]:

config = ExecutionCostsConfig(spread_rate=0.0002, commission_rate=0.0001, slippage_rate=0.00005, turnover_penalty=0.0001)
state = initialize_portfolio(initial_balance=100000.0)

# Apertura long a mid=100
open_trade = execute_order(timestamp="2024-01-01 10:00", mid_price=100.0, previous_position=0, target_position=1, config=config)
state = update_portfolio(previous_state=state, execution=open_trade, mid_price=100.0)
print(f"Dopo apertura: position={state.position} cash={state.cash:.6f} equity={state.equity:.6f}")

# Chiusura a mid=101
close_trade = execute_order(timestamp="2024-01-01 10:01", mid_price=101.0, previous_position=1, target_position=0, config=config)
state = update_portfolio(previous_state=state, execution=close_trade, mid_price=101.0)
print(f"Dopo chiusura: position={state.position} cash={state.cash:.6f} equity={state.equity:.6f}")

# Calcolo manuale indipendente dell'equity attesa
open_cost = 100.0 * (0.0002 / 2.0 + 0.00005 + 0.0001 + 0.0001)
close_cost = 101.0 * (0.0002 / 2.0 + 0.00005 + 0.0001 + 0.0001)
open_exec = 100.0 * (1.0 + 0.0002 / 2.0 + 0.00005)
close_exec = 101.0 * (1.0 - 0.0002 / 2.0 - 0.00005)
expected_equity = 100000.0 - open_exec - open_cost + close_exec - close_cost

print(f"\nEquity attesa (calcolo manuale): {expected_equity:.6f}")
print(f"Equity ottenuta dal codice:      {state.equity:.6f}")
print(f"Differenza: {abs(state.equity - expected_equity):.2e}")
assert state.position == 0 and state.entry_price is None
import math
assert math.isclose(state.equity, expected_equity, rel_tol=1e-12)
print("\nOK: equity esattamente uguale al calcolo manuale (criterio di completamento verificato).")


Dopo apertura: position=1 cash=99899.950000 equity=99999.950000
Dopo chiusura: position=0 cash=100000.899500 equity=100000.899500

Equity attesa (calcolo manuale): 100000.899500
Equity ottenuta dal codice:      100000.899500
Differenza: 0.00e+00

OK: equity esattamente uguale al calcolo manuale (criterio di completamento verificato).


### Round-trip short: `0 → -1 → 0`

Simmetricamente, si verifica un'apertura short a mid=100 e chiusura a mid=99 (prezzo favorevole per lo short).


In [11]:

state = initialize_portfolio(initial_balance=100000.0)

open_trade = execute_order(timestamp="2024-01-01 10:00", mid_price=100.0, previous_position=0, target_position=-1, config=config)
state = update_portfolio(previous_state=state, execution=open_trade, mid_price=100.0)

close_trade = execute_order(timestamp="2024-01-01 10:01", mid_price=99.0, previous_position=-1, target_position=0, config=config)
state = update_portfolio(previous_state=state, execution=close_trade, mid_price=99.0)

open_cost = 100.0 * (0.0002 / 2.0 + 0.00005 + 0.0001 + 0.0001)
close_cost = 99.0 * (0.0002 / 2.0 + 0.00005 + 0.0001 + 0.0001)
open_exec = 100.0 * (1.0 - 0.0002 / 2.0 - 0.00005)
close_exec = 99.0 * (1.0 + 0.0002 / 2.0 + 0.00005)
expected_equity_short = 100000.0 + open_exec - open_cost - close_exec - close_cost

print(f"Equity attesa (short round-trip): {expected_equity_short:.6f}")
print(f"Equity ottenuta dal codice:       {state.equity:.6f}")
assert math.isclose(state.equity, expected_equity_short, rel_tol=1e-12)
print("\nOK: round-trip short verificato con precisione floating-point.")


Equity attesa (short round-trip): 100000.900500
Equity ottenuta dal codice:       100000.900500

OK: round-trip short verificato con precisione floating-point.


### Verifica del mantenimento di posizione (hold)

Si verifica che mantenere una posizione long (`1 → 1`) senza chiuderla aggiorni correttamente solo l'unrealized P&L, senza generare nuovi costi di transazione.


In [12]:

state = initialize_portfolio(initial_balance=100000.0)
open_trade = execute_order(timestamp="2024-01-01 10:00", mid_price=100.0, previous_position=0, target_position=1, config=config)
state = update_portfolio(previous_state=state, execution=open_trade, mid_price=100.0)

costi_dopo_apertura = state.cumulative_costs

hold_trade = execute_order(timestamp="2024-01-01 10:01", mid_price=102.0, previous_position=1, target_position=1, config=config)
state = update_portfolio(previous_state=state, execution=hold_trade, mid_price=102.0)

print(f"Posizione mantenuta: {state.position}")
print(f"Unrealized P&L a mid=102 (entry=100.015): {state.unrealized_pnl:.6f}")
print(f"Costi cumulati invariati: {state.cumulative_costs == costi_dopo_apertura}")
assert state.position == 1
assert state.unrealized_pnl > 0.0
assert state.cumulative_costs == costi_dopo_apertura
print("\nOK: hold non genera costi aggiuntivi, solo variazione dell'unrealized P&L.")


Posizione mantenuta: 1
Unrealized P&L a mid=102 (entry=100.015): 1.985000
Costi cumulati invariati: True

OK: hold non genera costi aggiuntivi, solo variazione dell'unrealized P&L.


## 5. Suite di test automatici

Sono stati implementati 13 test dedicati all'Execution Engine e al Portfolio, distribuiti su tre file.

### `tests/test_execution.py` — 5 test

| Test | Cosa verifica |
|---|---|
| `test_buy_execution_price_is_worse_than_mid` | Il prezzo di acquisto è sempre penalizzato sopra il mid |
| `test_sell_execution_price_is_worse_than_mid` | Il prezzo di vendita è sempre penalizzato sotto il mid |
| `test_no_trade_keeps_mid_price` | Nessun trade (`trade_size=0`) restituisce esattamente il mid |
| `test_turnover_for_flip_is_two` | Un flip long→short produce turnover=2 |
| `test_total_cost_is_sum_of_components` | Il costo totale è la somma esatta delle quattro componenti |

### `tests/test_portfolio.py` — 3 test

| Test | Cosa verifica |
|---|---|
| `test_manual_long_roundtrip_matches_expected_equity` | **Criterio di completamento**: round-trip long produce l'equity calcolata manualmente |
| `test_manual_short_roundtrip_matches_expected_equity` | **Criterio di completamento**: round-trip short produce l'equity calcolata manualmente |
| `test_hold_long_updates_unrealized_pnl` | Il mantenimento di una posizione aggiorna solo l'unrealized P&L |

### `tests/test_state_frame_integration.py` — 2 test

| Test | Cosa verifica |
|---|---|
| `test_state_frame_reflects_simulated_positions` | Le feature di stato riflettono correttamente le posizioni simulate |
| `test_future_append_holds_with_real_state_frame` | Il criterio future-append della Fase 3 vale anche con stato reale (non stub) |

### Risultato dell'esecuzione completa del progetto

Eseguendo `pytest -v` sull'intero repository (Fase 3 + Fase 4 insieme):

```
tests/test_execution.py::test_buy_execution_price_is_worse_than_mid PASSED
tests/test_execution.py::test_sell_execution_price_is_worse_than_mid PASSED
tests/test_execution.py::test_no_trade_keeps_mid_price PASSED
tests/test_execution.py::test_turnover_for_flip_is_two PASSED
tests/test_execution.py::test_total_cost_is_sum_of_components PASSED
tests/test_features.py::test_selected_feature_columns_exist PASSED
tests/test_features.py::test_warmup_flag_becomes_true_after_indicator_history PASSED
tests/test_features.py::test_missing_and_infinite_checks_ignore_warmup_after_drop PASSED
tests/test_features.py::test_training_only_normalization_centers_training_split PASSED
tests/test_features.py::test_correlation_filter_drops_later_duplicate_feature PASSED
tests/test_features.py::test_strict_feature_contract_rejects_wrong_phase_windows PASSED
tests/test_no_leakage.py::test_future_append_does_not_change_past_features PASSED
tests/test_no_leakage.py::test_future_append_does_not_change_past_normalized_values_when_train_stats_are_reused PASSED
tests/test_portfolio.py::test_manual_long_roundtrip_matches_expected_equity PASSED
tests/test_portfolio.py::test_manual_short_roundtrip_matches_expected_equity PASSED
tests/test_portfolio.py::test_hold_long_updates_unrealized_pnl PASSED
tests/test_state_frame_integration.py::test_state_frame_reflects_simulated_positions PASSED
tests/test_state_frame_integration.py::test_future_append_holds_with_real_state_frame PASSED

============================== 18 passed in 0.41s ==============================
```

Tutti e 18 i test del progetto (Fase 3 + Fase 4) passano contemporaneamente, confermando che la nuova Execution Engine non ha introdotto regressioni nel modulo di feature engineering.


## 6. Collegamento con la Fase 3: `simulation.py`

### Il problema da risolvere

`build_state_frame` in `features.py` accetta un parametro opzionale `state_frame` con due colonne (`current_position`, `unrealized_return`). Prima della Fase 4 questo era sempre uno stub a zero. Ora che `portfolio.py` produce `position` e `unrealized_pnl` reali barra per barra, serve una funzione che li traduca in questo formato.

### Definizione di `unrealized_return`

`unrealized_pnl` è un valore assoluto in valuta; la feature richiesta è invece un **ritorno relativo**, normalizzato sull'equity corrente:

\\[ \\text{unrealized\\_return}_t = \\frac{\\text{unrealized\\_pnl}_t}{\\text{equity}_t} \\]

Questa scelta rende la feature comparabile tra episodi con capitale iniziale diverso, coerentemente con le altre feature della Fase 3 (già tutte in scala percentuale o normalizzata su ATR).


In [13]:

from collections.abc import Sequence
import pandas as pd


def simulate_portfolio_over_series(
    mid_prices: pd.Series,
    target_positions: Sequence[int],
    config: ExecutionCostsConfig,
    initial_balance: float,
) -> pd.DataFrame:
    """
    Simula l'Execution Engine barra per barra su una sequenza di decisioni
    di posizione già note (es. policy fissa o backtest storico).

    Ogni riga dipende esclusivamente dal prezzo corrente e dallo stato del
    portafoglio calcolato sulle barre precedenti: nessuna informazione
    futura entra nel calcolo, quindi il risultato è causale per costruzione
    e compatibile con il criterio future-append della Fase 3.
    """
    if len(mid_prices) != len(target_positions):
        raise ValueError("mid_prices e target_positions devono avere la stessa lunghezza.")

    state = initialize_portfolio(initial_balance=initial_balance)
    positions, unrealized_returns, equities = [], [], []

    for timestamp, mid_price, target_position in zip(mid_prices.index, mid_prices.to_numpy(), target_positions):
        execution = execute_order(
            timestamp=timestamp, mid_price=float(mid_price),
            previous_position=state.position, target_position=int(target_position), config=config,
        )
        state = update_portfolio(previous_state=state, execution=execution, mid_price=float(mid_price))

        positions.append(state.position)
        equities.append(state.equity)
        unrealized_returns.append(state.unrealized_pnl / state.equity if state.equity != 0.0 else 0.0)

    return pd.DataFrame(
        {"current_position": positions, "unrealized_return": unrealized_returns, "equity": equities},
        index=mid_prices.index,
    )


### Perché il test future-append passa anche con lo stato reale

Il test `test_future_append_holds_with_real_state_frame` ricostruisce **l'intera catena**: simulazione del portafoglio, esecuzione ordini, e costruzione feature, sia su `history` sia su `history + future`. Se una qualunque parte della catena fosse anche minimamente non causale, il confronto fallirebbe immediatamente, perché la simulazione userebbe informazioni diverse nei due scenari.

Il fatto che il test passi conferma che l'intera pipeline — decisione di posizione, esecuzione, aggiornamento portafoglio, costruzione feature — è causale end-to-end, non solo il modulo `features.py` isolato dalla Fase 3.


## 7. Sintesi finale della Fase 4

| Attività richiesta | Stato | Evidenza |
|---|---|---|
| Stabilire il prezzo di esecuzione | ✅ Completato | `get_execution_price`, penalizzato rispetto al mid in base al lato del trade |
| Simulare lo spread | ✅ Completato | `spread_rate/2` applicato simmetricamente, in assenza di colonne Bid/Ask nel dataset |
| Aggiungere commissioni | ✅ Completato | `commission_rate * notional` in `calculate_transaction_costs` |
| Aggiungere slippage | ✅ Completato | `slippage_rate` incorporato in `get_execution_price`, sempre sfavorevole al trader |
| Calcolare il turnover | ✅ Completato | `get_turnover`, verificato uguale a 2 per un flip long→short |
| Gestire capitale ed equity | ✅ Completato | `PortfolioState`, `equity = cash + position * mid_price` |
| Calcolare P&L realizzato e non realizzato | ✅ Completato | Separazione esplicita in `update_portfolio` e `calculate_unrealized_pnl` |
| Verificare manualmente long e short | ✅ Completato | 2 round-trip completi verificati con `rel_tol=1e-12` |

### Criterio di completamento verificato

> Una sequenza di azioni costruita manualmente deve produrre esattamente l'equity attesa.

Verificato sia nel notebook (sezione 4, calcolo manuale indipendente) sia nella suite di test automatici (`test_manual_long_roundtrip_matches_expected_equity`, `test_manual_short_roundtrip_matches_expected_equity`), con tolleranza di precisione floating-point.

### Verifica di integrità del repository

Tutti i sei file della Fase 4 sono stati confermati presenti nel repository GitHub del progetto e il loro contenuto corrisponde esattamente al codice progettato e discusso, senza discrepanze. L'esecuzione indipendente dell'intera suite di test (18 test totali, Fase 3 + Fase 4) confirma che il sistema è integro e privo di regressioni.

### Prossimi passi

1. **Script dimostrativo** (`scripts/demo_execution_trace.py`): una traccia stampata passo-passo di una sequenza di operazioni, utile come artefatto visivo per il report di progetto, oltre ai test automatici.
2. **Test dei casi limite residui**: `0 → 0` (nessuna variazione), flip diretto `1 → -1` con verifica esplicita del nuovo `entry_price`.
3. **Fase 5 — Ambiente RL**: sostituire la simulazione batch (`simulate_portfolio_over_series`) con una versione single-step, chiamata iterativamente dentro il loop dell'environment Gym/Gymnasium, dove le `target_positions` non sono più note a priori ma generate dall'agente durante l'interazione.
